In [ ]:

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

_nlp = None


def get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("en_core_web_sm")
    return _nlp


def extract_entities(text):
    """Extract named entities from text."""
    doc = get_nlp()(text)

    entities = []
    seen = set()

    for ent in doc.ents:
        item = (ent.text.strip(), ent.label_)

        if item[0] and item not in seen:
            seen.add(item)
            entities.append({
                "text": item[0],
                "label": item[1]
            })

    return entities


def extract_keywords(text, top_k=10):
    """Rank single-word and two-word keywords using TF-IDF."""
    text = text.strip()

    if not text:
        return []

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )

    try:
        matrix = vectorizer.fit_transform([text])
    except ValueError:
        return []

    scores = matrix.toarray()[0]
    terms = vectorizer.get_feature_names_out()

    ranked = sorted(
        zip(terms, scores),
        key=lambda item: item[1],
        reverse=True
    )

    return [
        {"keyword": term, "score": round(float(score), 4)}
        for term, score in ranked[:top_k]
    ]


def analyze_document(text):
    """Return a combined NLP analysis."""
    doc = get_nlp()(text)

    return {
        "word_count": len(text.split()),
        "sentence_count": len(list(doc.sents)),
        "keywords": extract_keywords(text),
        "entities": extract_entities(text)
    }
